# hrono — repo analysis notebook

Purpose: derive `chunking.toml`'s `chunk_size`/`chunk_overlap` per category from
measurement instead of a guessed number, per the discussion in the project
conversation log — we don't have access to real enterprise repos, so this uses
large, popular open-source repos as a structural proxy for "enterprise scale,
multiple languages, real production patterns."

Repos picked (one per major backend/frontend language, all large + widely used):
- `spring-projects/spring-boot` (Java)
- `django/django` (Python)
- `nestjs/nest` (TypeScript — explicitly an enterprise-style Node framework)
- `moby/moby` (Go)

This is a one-time snapshot for analysis, not a live source — shallow-cloned
(`--depth 1`) into `.hrono/analysis-cache/`, separate from the real `hrono.toml`
source cache so it never gets mixed into the actual knowledge tree.

Step 1: this notebook is file-level only (size/line distribution per category).
Function/class-level boundary analysis (the actual containment-rate metric
discussed) is the next step, once these numbers show whether file size alone
is a reasonable proxy or whether we need to go straight to AST parsing.

In [ ]:
import subprocess
from pathlib import Path

APP_DIR = Path.cwd()
ANALYSIS_CACHE = APP_DIR / ".hrono" / "analysis-cache"
ANALYSIS_CACHE.mkdir(parents=True, exist_ok=True)

REPOS = [
    ("https://github.com/spring-projects/spring-boot.git", "java"),
    ("https://github.com/django/django.git", "python"),
    ("https://github.com/nestjs/nest.git", "typescript"),
    ("https://github.com/moby/moby.git", "go"),
]


def shallow_clone(url: str) -> Path:
    name = url.rsplit("/", 1)[-1].removesuffix(".git")
    dest = ANALYSIS_CACHE / name
    if dest.exists():
        print(f"already cloned: {name}")
    else:
        print(f"cloning {name} (depth 1)")
        subprocess.run(["git", "clone", "--depth", "1", url, str(dest)], check=True, capture_output=True, text=True)
    return dest


# --- run: shallow-clone every repo in REPOS ---
repo_paths = []
for url, language in REPOS:
    path = shallow_clone(url)
    repo_paths.append((path, language))

for path, language in repo_paths:
    print(f"{language}: {path}")


Step 2: walk each repo, apply the real `filter.toml` (same filter the actual
pipeline uses, so these stats reflect what would actually get chunked) and tag
each surviving file with its `chunking.toml` category.

In [ ]:
import tomllib

with open(APP_DIR / "filter.toml", "rb") as f:
    filter_config = tomllib.load(f)
with open(APP_DIR / "chunking.toml", "rb") as f:
    chunking_config = tomllib.load(f)

_max_bytes = filter_config["limits"]["max_size_mb"] * 1024 * 1024
_excluded_patterns = filter_config["exclude"]["patterns"]
_sniff_enabled = filter_config["sniff"]["enabled"]
_default_category = chunking_config["default"]["category"]
_ext_to_category = chunking_config["extensions"]

SNIFF_BYTES = 8000


def is_binary(path: Path) -> bool:
    with open(path, "rb") as f:
        head = f.read(SNIFF_BYTES)
    return b"\x00" in head


def should_chunk(path: Path, root: Path) -> bool:
    rel = path.relative_to(root)
    if any(rel.full_match(pattern) for pattern in _excluded_patterns):
        return False
    size = path.stat().st_size
    if size == 0 or size > _max_bytes:
        return False
    if _sniff_enabled and is_binary(path):
        return False
    return True


def category_for(path: Path) -> str:
    return _ext_to_category.get(path.suffix.lower(), _default_category)


# --- run: collect (repo, language, category, extension, size_bytes, line_count) per kept file ---
records = []
for repo_path, language in repo_paths:
    for path in repo_path.rglob("*"):
        if not path.is_file() or not should_chunk(path, repo_path):
            continue
        size = path.stat().st_size
        try:
            lines = sum(1 for _ in path.open("r", encoding="utf-8", errors="ignore"))
        except OSError:
            continue
        records.append({
            "repo": repo_path.name,
            "language": language,
            "category": category_for(path),
            "extension": path.suffix.lower(),
            "size_bytes": size,
            "lines": lines,
        })

print(f"{len(records)} files kept across {len(repo_paths)} repos")
for repo_path, _ in repo_paths:
    n = sum(1 for r in records if r["repo"] == repo_path.name)
    print(f"  {repo_path.name}: {n} files")


Step 3: size distribution per category (bytes) — percentiles + a bucket
histogram, aggregated across all four repos. This is the number that should
drive `chunking.toml`'s `chunk_size`: e.g. if p90 for `code` files is ~1800
bytes, a 1600-char `chunk_size` is already splitting more than 10% of files at
least once, regardless of separator quality.

In [ ]:
def percentile(values: list[int], p: float) -> float:
    s = sorted(values)
    idx = min(len(s) - 1, int(round(p / 100 * (len(s) - 1))))
    return s[idx]


def print_distribution(label: str, sizes: list[int]) -> None:
    if not sizes:
        print(f"{label}: no files")
        return
    print(f"{label}: n={len(sizes)} min={min(sizes)} "
          f"p50={percentile(sizes, 50):.0f} p75={percentile(sizes, 75):.0f} "
          f"p90={percentile(sizes, 90):.0f} p95={percentile(sizes, 95):.0f} "
          f"p99={percentile(sizes, 99):.0f} max={max(sizes)} "
          f"mean={sum(sizes) / len(sizes):.0f}")


BUCKETS = [500, 1_000, 2_000, 4_000, 8_000, 16_000, 32_000, float("inf")]


def print_histogram(sizes: list[int]) -> None:
    lower = 0
    for upper in BUCKETS:
        count = sum(1 for s in sizes if lower <= s < upper)
        pct = 100 * count / len(sizes)
        label = f"{lower}-{upper if upper != float('inf') else 'inf'}"
        print(f"    {label:>12}: {count:5d}  {'#' * int(pct)} {pct:.1f}%")
        lower = upper


# --- run: per-category distribution across all repos, then per repo per category ---
categories = sorted({r["category"] for r in records})

print("=== aggregated across all repos ===")
for cat in categories:
    sizes = [r["size_bytes"] for r in records if r["category"] == cat]
    print_distribution(cat, sizes)
    print_histogram(sizes)

print()
print("=== per repo ===")
for repo_path, language in repo_paths:
    print(f"--- {repo_path.name} ({language}) ---")
    for cat in categories:
        sizes = [r["size_bytes"] for r in records if r["repo"] == repo_path.name and r["category"] == cat]
        if sizes:
            print_distribution(f"  {cat}", sizes)


Step 4: minimum `chunk_size` needed so that a given fraction of files are a
single whole chunk (no split at all) — i.e. `percentile(sizes, threshold)` per
category. Shown aggregated and per language, since Step 3 already showed size
varies a lot by language within the same category.

In [ ]:
THRESHOLDS = [70, 80, 90]


def min_size_table(sizes_by_key: dict[str, list[int]]) -> None:
    header = "".join(f"{f'p{t}':>10}" for t in THRESHOLDS)
    print(f"{'':20}{header}   n")
    for key, sizes in sizes_by_key.items():
        if not sizes:
            continue
        row = "".join(f"{percentile(sizes, t):>10.0f}" for t in THRESHOLDS)
        print(f"{key:20}{row}   {len(sizes)}")


print("=== aggregated across all repos (bytes) ===")
min_size_table({cat: [r["size_bytes"] for r in records if r["category"] == cat] for cat in categories})

print()
print("=== per language (bytes) ===")
for language in sorted({r["language"] for r in records}):
    print(f"--- {language} ---")
    min_size_table({
        cat: [r["size_bytes"] for r in records if r["language"] == language and r["category"] == cat]
        for cat in categories
    })


Step 5: find the "critical point" on the size-coverage curve automatically —
the `chunk_size` beyond which covering more files costs disproportionately
more bytes. Kneedle method (Satopaa et al. 2011): plot fraction of files
covered (x, 0-1) against bytes needed (y, normalized 0-1), pick the point of
maximum distance below the line connecting the curve's endpoints — that's
where local marginal cost (extra bytes per extra 1% coverage) crosses the
average marginal cost for the whole range.

First pass (linear y) put the knee at ~97% coverage — too far right to trust:
file sizes are heavily right-skewed (a handful of huge outlier files, e.g. a
407KB code file), so a *linear* y-axis lets those rare extreme values set the
scale for the whole curve, dragging the "average slope" line up and pushing
the knee out to cover them too. Standard fix: run Kneedle in *log*-byte space
instead (file sizes are closer to log-normal), so the tail no longer dominates
the distance calculation. Comparing both below.

In [ ]:
import math


def find_knee(sizes: list[int], resolution: int = 200, log_scale: bool = True) -> dict:
    s = sorted(sizes)
    ps = [100 * i / resolution for i in range(1, resolution + 1)]
    qs = [percentile(s, p) for p in ps]
    x = [p / 100 for p in ps]
    vals = [math.log(q) if q > 0 else 0.0 for q in qs] if log_scale else qs
    v_min, v_max = vals[0], vals[-1]
    y = [(v - v_min) / (v_max - v_min) for v in vals] if v_max > v_min else [0.0] * len(vals)
    diffs = [yi - xi for xi, yi in zip(x, y)]
    knee_i = min(range(len(diffs)), key=lambda i: diffs[i])
    return {"chunk_size": qs[knee_i], "coverage_pct": ps[knee_i], "n": len(s)}


# --- run: knee point per category, aggregated across all repos, linear vs log y-axis ---
print(f"{'category':10}{'linear size':>14}{'linear cov':>12}{'log size':>12}{'log cov':>10}")
knees = {}
for cat in categories:
    sizes = [r["size_bytes"] for r in records if r["category"] == cat]
    linear = find_knee(sizes, log_scale=False)
    log = find_knee(sizes, log_scale=True)
    knees[cat] = log
    print(f"{cat:10}{linear['chunk_size']:>14.0f}{linear['coverage_pct']:>11.1f}%"
          f"{log['chunk_size']:>12.0f}{log['coverage_pct']:>9.1f}%")


## Conclusion

**Decision: `chunk_size_kb` = 10 for `code`, 10 for `docs`, 2 for `data`.
No `chunk_overlap` — the config field was removed entirely.**
(applied in `../chunking.toml`, splitter logic in `../research.ipynb`).

**Why these sizes, not something else:**
- Self-containment (one chunk = one whole file, ideally one whole
  function/section) is the property we actually care about — not an
  arbitrary character count. So the target size should be *derived* from
  where real files stop being "mostly small" and start being "occasional
  huge outlier," not picked by feel.
- That transition point is the knee of the size-coverage curve: fraction of
  files covered (x) vs. bytes needed (y). Found it with Kneedle (Satopaa et
  al. 2011) — the point of max distance below the line connecting the
  curve's two ends, i.e. where marginal bytes-per-extra-file-covered crosses
  the *average* marginal cost for the whole distribution.
- First attempt (linear byte scale) put the knee at 97-98% coverage — not
  trustworthy: file sizes are heavily right-skewed (a few outlier files up
  to ~1MB), so a linear y-axis lets those rare extremes set the scale for
  the whole curve. Fix: Kneedle in **log-byte space** instead (file sizes
  are closer to log-normal) — gave code=10122B/91.5%, docs=10190B/90.5%,
  data=2280B/87.0%, consistent with the plain p90 percentiles (8924/9698/3556),
  which is what made the log-scale result trustworthy. Rounded to clean KB
  (10/10/2) since the rounding error (<2%) is inside the noise of the estimate.

**Why no `chunk_overlap` (this took two tries to get right):**
- First pass measured "how long is a typical function signature / doc
  heading line" (p90 ~80/72 chars) on the theory that overlap should carry
  the enclosing unit's header back into a continuation chunk. Wrong: the
  actual splitter only ever copies the trailing N *characters* of the
  previous chunk (`current[-chunk_overlap:]`), which can't reach a
  signature line that's typically much further from the cut than N
  characters, for anything big enough to actually need splitting.
- Second pass reconsidered from the mechanism itself: `merge_pieces` never
  truncates a piece — it only starts a new chunk *between* two whole
  pieces from `split_text`, and every piece is the product of a real
  separator (newline/comma/space) *except* when a single token doesn't
  shrink below `chunk_size` even after splitting on spaces (e.g. a giant
  unbroken blob). That's the only case where a boundary can fall mid-word —
  and it needs correct reassembly of the raw-char pieces, not copied bytes.
  Checked our own real corpus for that case: the only file with a
  pathologically long "line" turned out to be `moby/daemon/testdata/stat`,
  a no-extension test fixture that fell into `docs` via the category
  default — not genuine content. Reinforces that the scenario is rare to
  nonexistent in real code/docs/data.
- While checking that path, found `merge_pieces`'s old join logic had a real
  bug: it joined raw-char fallback pieces with `"\n"` unconditionally,
  turning e.g. `"HELLOWORLD"` into `"H\nE\nL\nL\nO..."` — corrupting the
  exact token it was supposed to preserve. Fixed by tracking which pieces
  came from the raw-char fallback and joining those (and only those) with
  `""` instead of `"\n"`. Verified byte-for-byte reconstruction on a
  synthetic oversized token.
- Net result: `chunk_overlap` wasn't a number to tune, it was solving a
  problem (mid-word cuts) that the splitter's own separator hierarchy
  already prevents except in one rare case — and that case needed a
  correctness fix, not a bigger config value. Removed the field.

**Still open**, tracked in `../TODO.md`: per-language chunk_size (code p70
ranged 1404B TypeScript to 5914B Python — one global number per category is
a simplification), and file-level size as a proxy for the real per-function
containment question (needs AST/tree-sitter, not just byte counts).

## Step 6: what should chunk_overlap actually cover?

Correction after checking the real implementation (`merge_with_overlap` in
`research.ipynb`): `chunk_overlap` copies the trailing **N characters** of the
previous merged chunk into the next one — `current[-chunk_overlap:]`. It is
*not* structure-aware. It cannot reliably carry a function's signature or a
doc's heading into a continuation chunk unless the split happens to land
within N characters of that line — and for anything big enough to actually
need splitting, the signature sits far upstream of the cut, not near it. So
sizing overlap off "how long is a typical signature/heading line" (attempted
below, then discarded) doesn't match what this mechanism can deliver.

What the trailing-char-window mechanism *does* deliver: local continuity at
the exact cut point — the next chunk isn't starting mid-statement/mid-sentence
with zero lead-in. The right measurement for that job is a generic one: how
many characters does a typical line take in that category, so overlap covers
roughly 2-3 lines of immediate lead-in context. Measured below.

In [ ]:
def collect_line_lengths_all(path: Path) -> list[int]:
    try:
        return [len(line) for line in path.read_text(encoding="utf-8", errors="ignore").splitlines() if line.strip()]
    except OSError:
        return []


# --- run: generic non-blank line length, per category, across all repos ---
line_lengths = {"code": [], "docs": [], "data": []}
for repo_path, language in repo_paths:
    for path in repo_path.rglob("*"):
        if not path.is_file() or not should_chunk(path, repo_path):
            continue
        cat = category_for(path)
        if cat in line_lengths:
            line_lengths[cat].extend(collect_line_lengths_all(path))

for cat, lengths in line_lengths.items():
    print_distribution(f"{cat} line length", lengths)
